# How many independent evaluation axioms are there?

This notebook counts how many independent evaluation axioms there are for an ensemble of $N$ classifiers. There are three sets of equations that define the logic of unsupervised evaluation for classifiers. The first two sets of equations (called an ideal in algebraic geometry) define the set of all possible evaluations for a test of size $Q$. These are the evaluations before we observe any test results.

The third set of equations use the counts of how the classifiers agreed and disagreed on the test. For a classification test with $R$ possible labels, there are $R^N$ ways that classifiers can do this when labeling a single item. Adding the third set of equations to the first two then gives the restricted set of the group evaluations that are **logically consistent** with the test results.

We will go over these set of equations and use linear algebra to count how many of these axioms are independent. Note that this is very different from asking if the classifiers where independent in their errors during a test. That is a different independence -- classifier independence. This notebook explores **axiom independence**.

In [1]:
import itertools, random
import sympy
from IPython.display import display,Math,Latex, HTML
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import ntqr
import ntqr.raxioms
sympy.init_session()
sympy.init_printing(latex_mode='equation',fontsize='12pt',print_builtin=False, use_unicode=True, use_latex='mathjax')

IPython console for SymPy 1.14.0 (Python 3.13.9-64-bit) (ground types: gmpy)

These commands were executed:
>>> from sympy import *
>>> x, y, z, t = symbols('x y z t')
>>> k, m, n = symbols('k m n', integer=True)
>>> f, g, h = symbols('f g h', cls=Function)
>>> init_printing()

Documentation can be found at https://docs.sympy.org/1.14.0/



## The simplex axioms

The **simplex axioms** are shown here. These are the equations stating that the count of all events conditioned on a true label must be equal to the appearance of that label in the answer key. This is a normalization of events statement. Given R labels and N classifiers making decisions, we have a complete description of their decision events ($R^N$ of them). The count of a label, $\ell$ , in the answer key is denoted by $Q_{\ell}$.

There are many of these simplex equations for an ensemble of size $N$, one for every subset of them and for every label. Why do we need all these spaces to talk about the group evaluations for the classifiers? What is a **group evaluation**? In essence, we want to know the partition of any decision event by true label. But we cannot just solve for these variables since it will admit too many partitions by true label. Some of these partitions will not marginalize properly when we look at subsets of the classifiers.

Let's count the simplex axioms,

\begin{equation}
\sum_{l=1}^{R} \sum_{m=1}^{N} \binom{N}{m} = R \left(2^N - 1 \right)
\end{equation}

Assuming that the answer key is fixed (not a variable), the number of variables in these equations is larger,

\begin{equation}
\sum_{l=1}^{R} \sum_{m=1}^{N} R^m \binom{N}{m} = R \left( (1+R)^N - 1 \right)
\end{equation}

Let's test the count of the variables

In [2]:
# This class is being deprecated in the upcoming version
labels = ('a','b','c')
classifiers = ('i','j','k')
mVars = ntqr.statistics.MClassifiersVariables(labels, classifiers)
mVars

MClassifiersVariables(('a', 'b', 'c'),('i', 'j', 'k'))

In [3]:
nVars = sum(len([value for value in label_vars.values()]) for msubset,decisions_by_true_label in mVars.label_responses.items()
   for decisions, label_vars in decisions_by_true_label.items())
nVars == 3*((1+3)**3 - 1)

True

In [4]:
3*((4)**3-1)

189

### The independence of the simplex axioms

Let's collect all the simplex axioms for the evaluation of $N$ classifiers. We'll check that the count we get for our working case of $N=3$ and $R=3$ has the expected count.

We then compute the rank of the matrix associated with the simplex axioms (they are all linear equations in the label response variables). Given how we constructed the simplex axioms, we expect all of them to be independent of each other. Is that so?

In [5]:
simplex_axioms = set(axiom for m in range(1,4)
                  for m_subset in itertools.combinations(classifiers, m)
                  for axiom in ntqr.raxioms.SimplexAxioms(labels, m_subset).axioms.values())
n_axioms = len(simplex_axioms) # returns 21
n_axioms == 3*(2**3-1)

True

In [6]:
# Let's look at one of the axioms to exhibit its simple linear structure
random.choice(list(simplex_axioms))

-Q_b + R_{a_{j},a_{k},b} + R_{a_{j},b_{k},b} + R_{a_{j},c_{k},b} + R_{b_{j},a_ ↪

↪ {k},b} + R_{b_{j},b_{k},b} + R_{b_{j},c_{k},b} + R_{c_{j},a_{k},b} + R_{c_{j ↪

↪ },b_{k},b} + R_{c_{j},c_{k},b}

In [7]:
# Let's use SymPy to check the independence of these equations, we expect the
# rank of the matrix of the linear coefficients to be 21.
#
# We need the vars
r_vars = [r_var for m in range(1,4)
          for m_subset in itertools.combinations(classifiers, m)
          for true_label in labels
          for r_var in ntqr.statistics.ResponseVariables(labels,m_subset).label_responses[true_label].values()]
# We checked before there are 189 of these variables
print(len(r_vars) == 189)
# These are the label response variables for subsets of the classifiers
random.choice(r_vars)

True


R_{b_{j},a_{k},c}

In [8]:
# Let's check the rank of the coefficient matrix
A, b = sympy.linear_eq_to_matrix(list(simplex_axioms), r_vars)
A

⎡0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  1  1  0  0  0  0  0  ↪
⎢                                                                              ↪
⎢0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  ↪
⎢                                                                              ↪
⎢0  0  0  0  0  0  0  0  0  1  1  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  ↪
⎢                                                                              ↪
⎢0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  1  1  0  0  0  0  0  0  0  0  ↪
⎢                                                                              ↪
⎢0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  ↪
⎢                                                                              ↪
⎢0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  1  ↪
⎢                                                                              ↪
⎢0  0  0  0  0  0  0  0  0  

In [9]:
A.rank() == 21

True

## The marginalization axioms

The evaluations defined by the simplex axioms alone are not correct. Since the axioms and the variables are unrelated, this implies that the set of possible evaluations is the product of the solutions in each simplex space alone. This is incorrect since the variables are algebraically distinct but their subscripts reflect that they can be marginalized counts. A point in a simplex for a subset must marginalize correctly to the value in simplexes with less classifiers.

We'll construct the axioms and check their independence with each other and with the simplex axioms.

In [10]:
# Let's get a peek at one of the axioms for the three classifiers
margAxioms = ntqr.raxioms.MarginalizationAxioms(labels, classifiers)
random.choice(margAxioms.axioms[random.choice(labels)])

R_{b_{i},a_{j},c_{k},b} + R_{b_{i},b_{j},c_{k},b} + R_{b_{i},c_{j},c_{k},b} -  ↪

↪ R_{b_{i},c_{k},b}

In [11]:
# Now let's collect the all. We include the marginalization of the single classifiers
# since the class for the axioms returns empty lists for them
all_marginalization_axioms = [axiom for m in range(1,len(classifiers)+1)
                              for m_subset in itertools.combinations(classifiers,m)
                              for label in labels
                              for axiom in ntqr.raxioms.MarginalizationAxioms(labels,m_subset).axioms[label]]
len(all_marginalization_axioms)

135

In [12]:
random.choice(all_marginalization_axioms)

R_{a_{i},b_{j},c_{k},c} + R_{b_{i},b_{j},c_{k},c} - R_{b_{j},c_{k},c} + R_{c_{ ↪

↪ i},b_{j},c_{k},c}

In [13]:
# Are they all independent?
A, b = sympy.linear_eq_to_matrix(list(all_marginalization_axioms), r_vars)
A.rank()

108

In [14]:
# What is the dimension of the space of all a-priori group evaluations
A, b = sympy.linear_eq_to_matrix(all_marginalization_axioms + list(simplex_axioms), r_vars)
A.rank()

111

In [15]:
# So the space of apriori group evaluations has dimension
189 - 111

78

## The observable axioms

The marginalization axioms involve sums of response variables with the same true label. Observable axioms involve sums across all true labels. If a pattern of decision by the classifiers occurs n times, that count is an unknown sum of non-negative integers over true labels.

An example is given by,
\begin{equation}
R_{c_{i},a_{j},c_{k},a} + R_{c_{i},a_{j},c_{k},b} + R_{c_{i},a_{j},c_{k},c} - R_{c_{i},a_{j},c_{k}}
\end{equation}

In [16]:
observable_axioms = [axiom for m in range(1,len(classifiers)+1)
                              for m_subset in itertools.combinations(classifiers,m)
                              for axiom in ntqr.raxioms.ObservableAxioms(labels,m_subset).axioms]
len(observable_axioms)

63

In [17]:
random.choice(observable_axioms)

R_{a_{i},c_{j},a} + R_{a_{i},c_{j},b} + R_{a_{i},c_{j},c} - R_{a_{i},c_{j}}

In [18]:
# How many are independent from themselves?
A, b = sympy.linear_eq_to_matrix(observable_axioms,r_vars)
A.rank()

63

In [19]:
# How about the marginalization + observable axioms?
A, b = sympy.linear_eq_to_matrix(all_marginalization_axioms + observable_axioms,r_vars)
A.rank()

135

In [20]:
A, b = sympy.linear_eq_to_matrix(all_marginalization_axioms + observable_axioms + list(simplex_axioms),r_vars)
A.rank()

137

Before we added the observable axioms, we had 111 independent axioms (simplex + marginalization). There are 63 observable axioms but they contribute only 26 independent axioms. This is a reduction in the dimension from 78 to 52.

The count for the number of independent observable axioms is equal to the number of ways three classifiers can agree/disagree on three labels minus one. This is intuitively correct since the observable equations for marginalized counts would follow from the marginalization axioms.